# 01. Ollama基礎：ローカルLLMをAPIで呼ぶ

## はじめに

**Ollama** は、ローカルマシン上でLLM（大規模言語モデル）を簡単に実行できるツールです。
クラウドAPIに依存せず、プライバシーを保ちながらLLMを活用できます。

このノートブックで学ぶこと：
- Ollamaサーバーへの接続確認
- テキスト生成（`generate`）の基本
- チャット形式（`chat`）の使い方
- ストリーミング出力
- 複数モデルのベンチマーク比較

**前提条件**：
- Ollamaがインストールされ、`ollama serve` が起動していること
- 少なくとも1つのモデルがダウンロード済みであること（例：`ollama pull qwen2.5:7b`）

In [ ]:
import sys
import os

# プロジェクトルートをパスに追加
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.ollama_client import OllamaClient, get_client
import json
import time

# デフォルトモデル
MODEL = "qwen2.5:7b"

print(f"使用モデル: {MODEL}")
print("セットアップ完了")

## 1. 接続確認

まずOllamaサーバーが起動しているか確認し、利用可能なモデルを一覧表示します。

In [ ]:
# クライアントを作成
client = get_client(model=MODEL)

# 接続確認
available = client.is_available()
print(f"Ollamaサーバー稼働中: {available}")

if not available:
    print("\n警告: Ollamaサーバーが起動していません。")
    print("ターミナルで 'ollama serve' を実行してください。")
else:
    # 利用可能なモデルを一覧表示
    models = client.list_models()
    print(f"\n利用可能なモデル ({len(models)}件):")
    for m in models:
        name = m.get('name', m.get('model', '不明'))
        size = m.get('size', 0)
        size_gb = size / (1024**3) if size else 0
        print(f"  - {name} ({size_gb:.1f} GB)" if size_gb > 0 else f"  - {name}")

## 2. テキスト生成（generate）

`generate()` メソッドは、プロンプトを受け取りテキストを生成します。
最もシンプルなLLMの使い方です。

In [ ]:
# 基本的なテキスト生成
prompt = "日本の首都はどこですか？"
print(f"プロンプト: {prompt}")
print("-" * 40)

response = client.generate(prompt)
print(f"応答: {response}")

In [ ]:
# temperatureによる出力の違いを比較
# temperature=0.0: 決定論的（常に同じ出力）
# temperature=1.5: 創造的（毎回異なる出力）

prompt = "短い俳句を一つ詠んでください。"
print(f"プロンプト: {prompt}")
print("=" * 50)

for temp in [0.0, 0.7, 1.5]:
    print(f"\n[temperature={temp}]")
    response = client.generate(prompt, temperature=temp)
    print(response)
    print("-" * 30)

print("\n観察: temperatureが低いほど一貫した応答、高いほど多様な応答になります")

## 3. チャット形式（chat）

`chat()` メソッドは、会話履歴を保持したマルチターン対話を実現します。
メッセージは `{"role": "user"|"assistant"|"system", "content": "..."}`の形式で渡します。

In [ ]:
# マルチターン会話の例
conversation_history = [
    {"role": "system", "content": "あなたは親切な日本語アシスタントです。簡潔に答えてください。"},
    {"role": "user", "content": "Pythonとは何ですか？"},
]

print("=== マルチターン会話 ===")
print(f"ユーザー: {conversation_history[-1]['content']}")

# 第1ターン
response1 = client.chat(conversation_history)
print(f"アシスタント: {response1}")
print()

# 会話履歴を更新して第2ターン
conversation_history.append({"role": "assistant", "content": response1})
conversation_history.append({"role": "user", "content": "Pythonの主な用途を3つ挙げてください。"})

print(f"ユーザー: {conversation_history[-1]['content']}")
response2 = client.chat(conversation_history)
print(f"アシスタント: {response2}")
print()

# 第3ターン
conversation_history.append({"role": "assistant", "content": response2})
conversation_history.append({"role": "user", "content": "それらの中で機械学習に最もよく使われるライブラリは？"})

print(f"ユーザー: {conversation_history[-1]['content']}")
response3 = client.chat(conversation_history)
print(f"アシスタント: {response3}")

## 4. ストリーミング出力

ストリーミングを使うと、生成されたテキストをリアルタイムで受け取れます。
長い応答の場合、ユーザー体験が大幅に向上します。

In [ ]:
# ストリーミング出力の例
prompt = "機械学習の歴史について200字程度で説明してください。"
print(f"プロンプト: {prompt}")
print("-" * 50)
print("[ストリーミング出力]")

start_time = time.time()
full_response = ""

for chunk in client.generate(prompt, stream=True):
    print(chunk, end='', flush=True)
    full_response += chunk

elapsed = time.time() - start_time
print(f"\n\n[完了] 生成時間: {elapsed:.2f}秒, 文字数: {len(full_response)}字")

## 5. モデル比較ベンチマーク

複数のモデルのパフォーマンスを比較します。
応答時間・トークン数・品質を評価しましょう。

In [ ]:
# ベンチマーク実行
print("ベンチマークを実行中... (しばらくお待ちください)")
print("=" * 60)

results = client.benchmark()

if results:
    # 結果をテーブル形式で表示
    print(f"{'モデル':<25} {'時間(秒)':<12} {'文字数':<10} {'文字/秒':<10}")
    print("-" * 60)
    for r in results:
        model_name = r.get('model', '不明')[:24]
        elapsed = r.get('elapsed', 0)
        chars = r.get('chars', len(r.get('response', '')))
        chars_per_sec = chars / elapsed if elapsed > 0 else 0
        status = r.get('status', 'ok')
        if status == 'ok':
            print(f"{model_name:<25} {elapsed:<12.2f} {chars:<10} {chars_per_sec:<10.1f}")
        else:
            print(f"{model_name:<25} {'エラー':<12} {'-':<10} {'-':<10}")
    print("-" * 60)
    print("\n各モデルの応答サンプル:")
    for r in results:
        if r.get('status') == 'ok':
            print(f"\n[{r.get('model', '不明')}]")
            response_preview = r.get('response', '')[:100]
            print(f"  {response_preview}..." if len(r.get('response', '')) > 100 else f"  {r.get('response', '')}")
else:
    print("ベンチマーク結果がありません。モデルがインストールされているか確認してください。")

## まとめ

このノートブックで学んだこと：

| 機能 | メソッド | 用途 |
|------|----------|------|
| 接続確認 | `is_available()` | サーバー起動確認 |
| モデル一覧 | `list_models()` | 利用可能モデル取得 |
| テキスト生成 | `generate(prompt)` | シンプルな生成 |
| チャット | `chat(messages)` | マルチターン対話 |
| ストリーミング | `generate(..., stream=True)` | リアルタイム出力 |
| ベンチマーク | `benchmark()` | モデル性能比較 |

**重要なポイント**：
- `temperature` パラメータで出力の多様性を制御できる（0.0=決定論的、1.5=創造的）
- チャット形式では会話履歴を明示的に管理する必要がある
- ストリーミングはUX向上に効果的
- モデルによって応答速度・品質・専門性が異なる

**次のステップ**：
- `02_prompt_engineering.ipynb` でプロンプト設計を学ぶ
- `03_rag_from_scratch.ipynb` でRAGパイプラインを実装する